In [ ]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Data Exploration

In [ ]:
df=pd.read_csv("Salary_dataset.csv")
df.head()

In [ ]:
df.columns

In [ ]:
df.shape

#here the number of rows are 30 and columns are 3 

In [ ]:
df.dtypes

# DATA CLEANING

In [ ]:
df= df.drop('Unnamed: 0', axis=1)
df

# here the shape of the dataset is now columns=2 and rows=30
# here we dropped the column Unnamed :0 as it was not needed

# Checking for missing values

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

### No missing values found

In [ ]:
df.duplicated().sum()

### No duplicates found. So, let's move towards understanding dataset

## Descriptive statistics

In [ ]:
df.describe()

In [ ]:
median_years = df['YearsExperience'].median()
median_salary = df['Salary'].median()
print(f"Median Years of Experience: {median_years}")
print(f"Median Salary: {median_salary}")

In [ ]:
mode_years = df['YearsExperience'].mode()[0]
mode_salary = df['Salary'].mode()[0]
print(f"Mode Years of Experience: {mode_years}")
print(f"Mode Salary: {mode_salary}")

## Descriptive statistics
* Average experience:  5 years	
* Avereage of Salary: 76004 rs
* Median Years of Experience: 4.8
* Median Salary: 65238 rs
* Most commonly occuring Years of Experience: 3 yrs
* Mode Salary: 37732 rs

## Conclusion
* The mean and mode are not matching as most people have 5 yrs of exp but common occuring is 3 ys, does this mean there are outlier? 
* The average salary is high, but the typical salary is about 65,238 INR. Many people earn 37,732 INR, which may be due to lower-paying jobs.

# Checking for Outliers

In [ ]:
for column in ['YearsExperience', 'Salary']:
    Q1, Q3 = df[column].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    outliers = df[(df[column] < Q1 - 1.5 * IQR) | (df[column] > Q3 + 1.5 * IQR)]
    print(f"Outliers in {column}:")
    print(outliers)

#### Ok, so there are no outliers so lets check for skewdness in histogram
# Univariate Analysis


In [ ]:
sns.histplot(df['YearsExperience'],bins=10,kde=True)

* Most employees have 3?5 years of experience.
* Fewer employees have 8+ years of experience.
* The distribution is slightly right-skewed.
* The peak frequency is around 4 years of experience.

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
sns.boxplot(x=df['YearsExperience'])
plt.subplot(1, 2, 2)
sns.boxplot(x=df['Salary'], color='red')
plt.title('Boxplot of Salary')
plt.xlabel('Salary')
plt.tight_layout()
plt.show()

### - most people are haveing more than four years of exp.
### - Most people are having salaray more than 60k above


### Many people are not new employers

# Bivariate Analysis

In [ ]:
sns.scatterplot(df,x='YearsExperience',y='Salary')

* the more experience the more the salary

In [ ]:
x=df.corr()
sns.heatmap(x,annot=True)


* ### Insights:
* Positive Trend: Salary increases as Years of Experience increase.
* Linear Relationship: The regression line shows a strong linear correlation.
* Consistent Growth: Salary growth is steady without significant fluctuations.
* Minor Deviations: Some points slightly deviate from the regression line, indicating outliers or individual variations.

In [ ]:
print("Skewness of YearsExperience:", df['YearsExperience'].skew())
print("Skewness of Salary:", df['Salary'].skew())
df['YearsExperience_log'] = np.log1p(df['YearsExperience'])  # log1p handles zero values
df['Salary_log'] = np.log1p(df['Salary'])
X = df[['YearsExperience_log']]
y = df['Salary_log']


In [ ]:
#As there is variation in the salary and Years of Experience 
#so lets perform minmax scaling and normalize it into the range 0 to 1 

from sklearn.preprocessing import MinMaxScaler
min=MinMaxScaler()
df['YearsExperience']=min.fit_transform(df[['YearsExperience']])

In [ ]:
df['YearsExperience']

In [ ]:
df['Salary']=min.fit_transform(df[['Salary']])
df['Salary']

# Log Transformation:
As the skewness of the dataset is not that good so for that we will perform the Logtransformation 

so that it will give more accuracy when is model is built

In [ ]:
from sklearn.preprocessing import FunctionTransformer

In [ ]:
trans=FunctionTransformer(np.log1p)
df['YearsExperience']=trans.fit_transform(df['YearsExperience'])

In [ ]:
sns.distplot(df['YearsExperience'],hist=False)

In [ ]:
sns.distplot(df['Salary'],hist=False)

In [ ]:
df['Salary']=trans.fit_transform(df['Salary'])
df['Salary']

In [ ]:
sns.distplot(df['Salary'],hist=False)

# Model Building and Evaluation

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures

* Feature Engineering: Polynomial Features
* Adding polynomial terms can help capture non-linear relationships.

In [ ]:
X = np.random.rand(100, 1)
y = 3 * X**2 + 2 * X + 1
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
plo = PolynomialFeatures()
X_train_pol = plo.fit_transform(X_train)
X_test_pol = plo.transform(X_test)

In [ ]:
print("Original X_train:")
print(X_train[:5])  # Display the first 5 rows of the original training data

print("\nTransformed X_train_pol:")
print(X_train_pol[:5])  # Display the first 5 rows of the transformed training data


In [ ]:
plt.scatter(X_train, y_train, color="blue", label="Original Data")
plt.scatter(X_train, X_train_pol[:, 1], color="red", label="Polynomial Transformation")
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression

linear = LinearRegression()
linear.fit(X_train_pol, y_train)
y_pre = linear.predict(X_test_pol)
r2 = r2_score(y_test, y_pre)
print(f"R? Score: {r2}")